# Immunogenicity tool test: Testing setttings for tools: NetMHC_II_pan_Immunogen
Author: Rebecka Antonsson\
Version: 1 (04-03-2026)

# Notebook explanation
Notebook to calculate the scores of the outputs from the tool BioPhi(OASis) with different settings.

Settings tested are:\
\
netMHC_II_pepl11_allel27_wind2_03_03_2026.csv
netMHC_II_pepl12_allel27_wind2_03_03_2026.csv
netMHC_II_pepl12_allel27_wind5_03_03_2026.csv
netMHC_II_pepl12_allel27_wind10_03_03_2026.csv
netMHC_II_pepl15_allel27_wind2_03_03_2026.csv
netMHC_II_pepl15_allel27_wind10_03_03_2026..csv
netMHC_II_pepl15_allel27_wind30_03_03_2026.csv



For each tool_output with different settings a ranking from 1 to 37 will be done, each antibody will hence be given a number 1 to 37
This ranking will be compared to the known/ clinically determined immunigenecity ranking of the antibodies. The clinical immunigenecity ranking is based on anti-drug antibody (ADA) data. The two nanobodies will be held outside the ranking since they have risk of behaving differently, and special intreset is taken in how the tool perform at prediction those.

For each setting three measurments on how good they perform will be calculated:
    1. Mean absolute rank error (MARE) for antibodies
    2. Spearman rank correlation
    3. Mean absolute rank error for the two nanobodies

1. MARE is just the absolute difference between the known rank of the antibody and the predicted rank
2. Spearman rank correlation is a statistical test that can compare two lists of ranking and tell how well they align. 1 is a perfect correlation
0 means no relation between the two variables (no correlation, random) and -1 is a perfect reversed correlation (very bad).
3. Same calculation as for 1, but separated from the rest. Here a separate ranking where the 2 nanobodies are included are performed, hence antobodies and nanobodies get a number 1 to 39 and the MARE for the nanobodies is calculated. 

In [1]:
import pandas as pd
from scipy.stats import spearmanr

In [2]:
# Load the tool outputs with the different setttings, store each in a separate pandas DataFrame
netMHC_II_immunogen_pepl11_allel27_wind2 = pd.read_csv("tool_outputs/netMHC_II_pepl11_allel27_wind2_03_03_2026.csv")
netMHC_II_immunogen_pepl12_allel27_wind2 = pd.read_csv("tool_outputs/netMHC_II_pepl12_allel27_wind2_03_03_2026.csv")
netMHC_II_immunogen_pepl12_allel27_wind5 = pd.read_csv("tool_outputs/netMHC_II_pepl12_allel27_wind5_03_03_2026.csv")
netMHC_II_immunogen_pepl12_allel27_wind10 = pd.read_csv("tool_outputs/netMHC_II_pepl12_allel27_wind10_03_03_2026.csv")
netMHC_II_immunogen_pepl15_allel27_wind2 = pd.read_csv("tool_outputs/netMHC_II_pepl15_allel27_wind2_03_03_2026.csv")
netMHC_II_immunogen_pepl15_allel27_wind10 = pd.read_csv("tool_outputs/netMHC_II_pepl15_allel27_wind10_03_03_2026.csv")
netMHC_II_immunogen_pepl15_allel27_wind30 = pd.read_csv("tool_outputs/netMHC_II_pepl15_allel27_wind30_03_03_2026.csv")
netMHC_II_immunogen_pepl15_allel7_wind5 = pd.read_csv("tool_outputs/netMHC_II_pepl15_allel7_wind5_03_03_2026.csv")
netMHC_II_immunogen_pepl11_allel7_wind2 = pd.read_csv("tool_outputs/netMHC_II_pepl11_allel7_wind2_03_03_2026.csv")


# Then also load the sequence table for each tool output. 
# This is because in the tool output of the scores each antibody sequence has just been given an number (1-39)
# With the sequence table I can map the antibody name back to the number it was given
seq_table_netMHC_II_immunogen_pepl11_allel27_wind2 = pd.read_csv("tool_outputs/sequence_table_netMHC_II_pepl11_allel27_wind2_03_03_2026.csv")
seq_table_netMHC_II_immunogen_pepl12_allel27_wind2 = pd.read_csv("tool_outputs/sequence_table_netMHC_II_pepl12_allel27_wind2_03_03_2026.csv")
seq_table_netMHC_II_immunogen_pepl12_allel27_wind5 = pd.read_csv("tool_outputs/sequence_table_netMHC_II_pepl12_allel27_wind5_03_03_2026.csv")
seq_table_netMHC_II_immunogen_pepl12_allel27_wind10 = pd.read_csv("tool_outputs/sequence_table_netMHC_II_pepl12_allel27_wind10_03_03_2026.csv")
seq_table_netMHC_II_immunogen_pepl15_allel27_wind2 = pd.read_csv("tool_outputs/sequence_table_netMHC_II_pepl15_allel27_wind2_03_03_2026.csv")
seq_table_netMHC_II_immunogen_pepl15_allel27_wind10 = pd.read_csv("tool_outputs/sequence_table_netMHC_II_pepl15_allel27_wind10_03_03_2026.csv")
seq_table_netMHC_II_immunogen_pepl15_allel27_wind30 = pd.read_csv("tool_outputs/sequence_table_netMHC_II_pepl15_allel27_wind30_03_03_2026.csv")
seq_table_netMHC_II_immunogen_pepl15_allel7_wind5 = pd.read_csv("tool_outputs/sequence_table_netMHC_II_pepl15_allel7_wind5_03_03_2026.csv")
seq_table_netMHC_II_immunogen_pepl11_allel7_wind2 = pd.read_csv("tool_outputs/sequence_table_netMHC_II_pepl11_allel7_wind2_03_03_2026.csv")

In [3]:
# Create dictionary with all df names too be able to loop through them
all_dfs = {
    'netMHC_II_immunogen_pepl11_allel27_wind2': netMHC_II_immunogen_pepl11_allel27_wind2,
    'netMHC_II_immunogen_pepl12_allel27_wind2': netMHC_II_immunogen_pepl12_allel27_wind2,
    'netMHC_II_immunogen_pepl12_allel27_wind5': netMHC_II_immunogen_pepl12_allel27_wind5,
    'netMHC_II_immunogen_pepl12_allel27_wind10': netMHC_II_immunogen_pepl12_allel27_wind10,
    'netMHC_II_immunogen_pepl15_allel27_wind2': netMHC_II_immunogen_pepl15_allel27_wind2,
    'netMHC_II_immunogen_pepl15_allel27_wind10': netMHC_II_immunogen_pepl15_allel27_wind10,
    'netMHC_II_immunogen_pepl15_allel27_wind30': netMHC_II_immunogen_pepl15_allel27_wind30,
    'netMHC_II_immunogen_pepl15_allel7_wind5': netMHC_II_immunogen_pepl15_allel7_wind5,
    'netMHC_II_immunogen_pepl11_allel7_wind2': netMHC_II_immunogen_pepl11_allel7_wind2
}

seq_tables = {
    'netMHC_II_immunogen_pepl11_allel27_wind2': seq_table_netMHC_II_immunogen_pepl11_allel27_wind2,
    'netMHC_II_immunogen_pepl12_allel27_wind2': seq_table_netMHC_II_immunogen_pepl12_allel27_wind2,
    'netMHC_II_immunogen_pepl12_allel27_wind5': seq_table_netMHC_II_immunogen_pepl12_allel27_wind5,
    'netMHC_II_immunogen_pepl12_allel27_wind10':seq_table_netMHC_II_immunogen_pepl12_allel27_wind10,
    'netMHC_II_immunogen_pepl15_allel27_wind2': seq_table_netMHC_II_immunogen_pepl15_allel27_wind2,
    'netMHC_II_immunogen_pepl15_allel27_wind10': seq_table_netMHC_II_immunogen_pepl15_allel27_wind10,
    'netMHC_II_immunogen_pepl15_allel27_wind30': seq_table_netMHC_II_immunogen_pepl15_allel27_wind30,
    'netMHC_II_immunogen_pepl15_allel7_wind5': seq_table_netMHC_II_immunogen_pepl15_allel7_wind5,
    'netMHC_II_immunogen_pepl11_allel7_wind2': seq_table_netMHC_II_immunogen_pepl11_allel7_wind2
    
}

In [4]:
# Sanity check
# Check so that all dataframes have all the 39 antibodies and the 27 alleles
for name, df in all_dfs.items():
    # check if unique values in seq # column is 39)
    if df['seq #'].nunique()!=39:
        print(f'{name, "does not have 39 antibodies/nanobodies"}')
    else:
        # check if number of unique valies in HLA allele column is 27)
        if df['allele'].nunique()==27:
            print(f'{name,"has 27 HLA alleles"}')
        elif df['allele'].nunique()==7:
            print(f'{name,"has 7 HLA alleles"}')
        elif df['allele'].nunique()==1:
            print(f'{name,"has 1 HLA allele"}')
        else:
            print(f'{name,"does not have 27 or 7 HLA alleles"}')


('netMHC_II_immunogen_pepl11_allel27_wind2', 'has 27 HLA alleles')
('netMHC_II_immunogen_pepl12_allel27_wind2', 'has 27 HLA alleles')
('netMHC_II_immunogen_pepl12_allel27_wind5', 'has 27 HLA alleles')
('netMHC_II_immunogen_pepl12_allel27_wind10', 'has 27 HLA alleles')
('netMHC_II_immunogen_pepl15_allel27_wind2', 'has 27 HLA alleles')
('netMHC_II_immunogen_pepl15_allel27_wind10', 'has 27 HLA alleles')
('netMHC_II_immunogen_pepl15_allel27_wind30', 'has 27 HLA alleles')
('netMHC_II_immunogen_pepl15_allel7_wind5', 'has 7 HLA alleles')
('netMHC_II_immunogen_pepl11_allel7_wind2', 'has 7 HLA alleles')


In [5]:
# Clean dataframes a bit, and instert the correct antibody names

for key in all_dfs:
    df = all_dfs[key]
    seq_table = seq_tables[key] 

    # Insert the sequence names (Antibody names) into the dataframe with scores, map by seq #, which exists in both dataframes
    df = df.merge(seq_table[['seq #', 'sequence name']], how='left')

    # Rename the column "sequence name" to "Antibody", 
    df.rename(columns={'sequence name': 'Antibody'}, inplace=True)

    # remove the two columns called seq # from the calc_rank_table
    df = df.drop(columns=['seq #'])

    # Place the anotbody columns as the first column beacuse its easier to read

    # Remove the last column and save it as a variable
    last_col = df.pop(df.columns[-1])
    # Insert the column as the first column
    df.insert(0, last_col.name, last_col)

    all_dfs[key] = df

In [6]:
all_dfs['netMHC_II_immunogen_pepl15_allel27_wind30'].head()

,Antibody,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_ba core,netmhciipan_ba ic50,netmhciipan_ba percentile,cd4episcore core,immunogenicity score,immunogenicity combined score
0,RITUXIMAB,AQIVLSQSPAILSAS,121,135,15,HLA-DRB1*13:02,254,0.04,IVLSQSPAI,6.93,0.04,IVLSQSPAI,97.6866,41.89464
1,RITUXIMAB,AQIVLSQSPAILSAS,121,135,15,HLA-DRB1*07:01,254,0.25,IVLSQSPAI,10.10,0.25,IVLSQSPAI,97.6866,41.89464
2,RITUXIMAB,AQIVLSQSPAILSAS,121,135,15,HLA-DRB1*09:01,254,0.29,IVLSQSPAI,19.24,0.29,IVLSQSPAI,97.6866,41.89464
3,SECUKINUMAB,TAVYYCVRDYYDILT,91,105,15,HLA-DQA1*01:01/DQB1*05:01,35,0.38,YYCVRDYYD,126.86,0.38,AVYYCVRDY,96.2055,55.28220
4,Caplacizumab,TAVYYCAAAGVRAED,91,105,15,HLA-DRB1*01:01,293,0.67,YCAAAGVRA,6.36,0.67,VYYCAAAGV,93.7309,45.89236


Here the score is calculated as nr of rows with netMHCpan percentile <=10, divided by total number of rows. Meaning that higher number indicated higher immunogenicity. The score can be described as the percentage of MHC-peptides that have a netMHCpan percentile below 10

In [7]:
# for all dfs in all_dfs, check the dtype of the column "immunogenicity score"
for key, df in all_dfs.items():
    print(f'{key}: {df["immunogenicity score"].dtype}')

netMHC_II_immunogen_pepl11_allel27_wind2: object
netMHC_II_immunogen_pepl12_allel27_wind2: object
netMHC_II_immunogen_pepl12_allel27_wind5: object
netMHC_II_immunogen_pepl12_allel27_wind10: object
netMHC_II_immunogen_pepl15_allel27_wind2: float64
netMHC_II_immunogen_pepl15_allel27_wind10: float64
netMHC_II_immunogen_pepl15_allel27_wind30: float64
netMHC_II_immunogen_pepl15_allel7_wind5: float64
netMHC_II_immunogen_pepl11_allel7_wind2: object


In [8]:
for key, df in all_dfs.items():
    if df["immunogenicity score"].dtype == "object":
        print(f"\nChecking {key}")
        
        non_numeric = df[
            pd.to_numeric(df["immunogenicity score"], errors="coerce").isna()
        ]["immunogenicity score"].unique()
        
        print("Non-numeric values found:")
        print(non_numeric[:20])  # show first 20 unique bad values

        print(df[df["immunogenicity score"] == "-"].shape[0])
        print(df.shape[0])


Checking netMHC_II_immunogen_pepl11_allel27_wind2
Non-numeric values found:
['-']
4563
114183

Checking netMHC_II_immunogen_pepl12_allel27_wind2
Non-numeric values found:
['-']
783
113670

Checking netMHC_II_immunogen_pepl12_allel27_wind5
Non-numeric values found:
['-']
378
45819

Checking netMHC_II_immunogen_pepl12_allel27_wind10
Non-numeric values found:
['-']
270
23166

Checking netMHC_II_immunogen_pepl11_allel7_wind2
Non-numeric values found:
['-']
1183
29603


In [9]:
# for all_dfs netMHC_II_immunogen_pepl12_allel27_wind10 print the rows where the immunogenicity score is equal to "-"
all_dfs['netMHC_II_immunogen_pepl12_allel27_wind10'][all_dfs['netMHC_II_immunogen_pepl12_allel27_wind10']['immunogenicity score'] == '-']

,Antibody,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_ba core,netmhciipan_ba ic50,netmhciipan_ba percentile,cd4episcore core,immunogenicity score,immunogenicity combined score
2276,MEPOLIZUMAB,KDTSRNQVVLTM,71,82,12,HLA-DRB3*02:02,318,15.0,TSRNQVVLT,9811.23,15.0,-,-,-
3368,DUPILUMAB,LEQPGGSLRLSC,11,22,12,HLA-DQA1*05:01/DQB1*03:01,512,21.0,QPGGSLRLS,3396.90,21.0,-,-,-
3482,RITUXIMAB,EDAATYYCQQWT,201,212,12,HLA-DQA1*01:01/DQB1*05:01,731,21.0,AATYYCQQW,6132.10,21.0,-,-,-
5154,MEPOLIZUMAB,KDTSRNQVVLTM,71,82,12,HLA-DRB1*13:02,318,29.0,TSRNQVVLT,6098.56,29.0,-,-,-
7129,RITUXIMAB,EDAATYYCQQWT,201,212,12,HLA-DQA1*03:01/DQB1*03:02,731,38.0,AATYYCQQW,6618.73,38.0,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23161,VISILIZUMAB,SSNPPTFGGGTK,211,222,12,HLA-DRB3*01:01,44,100.0,SNPPTFGGG,40639.83,100.0,-,-,-
23162,VISILIZUMAB,SSNPPTFGGGTK,211,222,12,HLA-DQA1*01:01/DQB1*05:01,44,100.0,SNPPTFGGG,41061.04,100.0,-,-,-
23163,VISILIZUMAB,SSNPPTFGGGTK,211,222,12,HLA-DRB3*02:02,44,100.0,SNPPTFGGG,41961.54,100.0,-,-,-
23164,VISILIZUMAB,SSNPPTFGGGTK,211,222,12,HLA-DRB1*03:01,44,100.0,PPTFGGGTK,42228.43,100.0,-,-,-


In [10]:
# Remove all rows where the immunogenicity score is equal to "-"
# And make the immunogenicity score column numeric instead of object
for key in all_dfs:
    df = all_dfs[key]
    df = df[df['immunogenicity score'] != '-']
    df['immunogenicity score'] = pd.to_numeric(df['immunogenicity score'], errors='coerce')
    all_dfs[key] = df

all_dfs['netMHC_II_immunogen_pepl12_allel27_wind10'].head()

C:\Users\rebbe\AppData\Local\Temp\ipykernel_12284\1374502062.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['immunogenicity score'] = pd.to_numeric(df['immunogenicity score'], errors='coerce')


,Antibody,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_ba core,netmhciipan_ba ic50,netmhciipan_ba percentile,cd4episcore core,immunogenicity score,immunogenicity combined score
0,RESLIZUMAB,KLLIYGANSLQT,161,172,12,HLA-DRB1*15:01,372,0.01,LLIYGANSL,57.66,0.01,KLLIYGANS,98.9982,43.07928
1,VISILIZUMAB,TLTISSLQPEDF,191,202,12,HLA-DQA1*05:01/DQB1*02:01,42,0.02,ISSLQPEDF,106.22,0.02,ISSLQPEDF,99.8586,56.74344
2,GALCANEZUMAB,TLTISSLQPEDF,191,202,12,HLA-DQA1*05:01/DQB1*02:01,419,0.02,ISSLQPEDF,106.22,0.02,ISSLQPEDF,99.8586,56.74344
3,USTEKINUMAB,TLTISSLQPEDF,191,202,12,HLA-DQA1*05:01/DQB1*02:01,508,0.02,ISSLQPEDF,106.22,0.02,ISSLQPEDF,99.8586,56.74344
4,IBALIZUMAB,LTISSVQAEDVA,201,212,12,HLA-DQA1*05:01/DQB1*02:01,155,0.02,ISSVQAEDV,104.14,0.02,ISSVQAEDV,98.4750,54.39


In [11]:
# Loop through all the dataframes, and calculate the mean of the immunogenicity score for each antibody.
# Overwrite the old dataframe and put the antibody name + the immunogen as the columns 
for key, df in all_dfs.items():
    score_df = (
        df.groupby('Antibody')['immunogenicity score']
        .mean()
        .reset_index(name='mean_immunogenicity_score')

    )

    all_dfs[key] = score_df # overwrite the old df 

In [12]:
# For all the dataframes in the all_dfs dictionary loop through them and make a copy 
# without the two rows where Antibody = Caplacizumab or Vobarilizumab
# Put the new dataframes into a new dictionary called all_dfs_AB

# we need to do this because we don't want these two nanobodies to be ranked, 
# but we also don't want them to mess with the ranking of the other antibodies

all_dfs_AB = {}

for name, df in all_dfs.items():
    # make a new dictionary with a copy of the dataframes but without the two nanobodies
    all_dfs_AB[name + "_AB"] = df.loc[~df['Antibody'].isin(['Caplacizumab', 'Vobarilizumab'])].copy()

# Check so it worked
all_dfs_AB.keys()
all_dfs_AB['netMHC_II_immunogen_pepl12_allel27_wind10_AB'].tail()

,Antibody,mean_immunogenicity_score
33,SECUKINUMAB,99.646009
34,TILDRAKIZUMAB,99.522000
35,USTEKINUMAB,99.431941
36,VEDOLIZUMAB,99.484070
37,VISILIZUMAB,99.537438


In [13]:
# Now add a column to the all_dfs_AB dataframe called ranked by tool, where the percent_immonogen values are ranked from highest to lowest
# Highest means highest immunogenicity, so thats 37.
for df in all_dfs_AB.values():
    df['ranked_by_tool'] = df['mean_immunogenicity_score'].rank(ascending=True, method='average')

# Do the same for the df includning the two nanobodies
for df in all_dfs.values():
    df['ranked_by_tool'] = df['mean_immunogenicity_score'].rank(ascending=True, method='average')

In [14]:
# Load the ADA ranking
ADA_rank = {
'BEZLOTOXUMAB':1,
'VISILIZUMAB':2,
'OMALIZUMAB':3,
'EVOLOCUMAB':4,
'SECUKINUMAB':5,
'DENOSUMAB':6,
'IBALIZUMAB':7,
'OCRELIZUMAB':8,
'FREMANEZUMAB':9,
'BASILIXIMAB':10,
'PALIVIZUMAB':11,
'CANAKINUMAB':12,
'ECULIZUMAB':13,
'BRODALUMAB':14,
'MEPOLIZUMAB':15,
'GUSELKUMAB':16,
'RESLIZUMAB':17,
'ALIROCUMAB':18,
'GALCANEZUMAB':19,
'VEDOLIZUMAB':20,
'EFALIZUMAB':21,
'TILDRAKIZUMAB':22,
'USTEKINUMAB':23,
'DUPILUMAB':24,
'ERENUMAB':25,
'SARILUMAB':26,
'NATALIZUMAB':27,
'Caplacizumab':28,
'LANADELUMAB':29,
'BUROSUMAB':30,
'BENRALIZUMAB':31,
'ADALIMUMAB':32,
'IXEKIZUMAB':33,
'RITUXIMAB':34,
'INFLIXIMAB':35,
'GOLIMUMAB':36,
'Vobarilizumab':37,
'BOCOCIZUMAB':38,
'ALEMTUZUMAB':39
}

# create antoher dictionary where the two nanobdoies are removed 
ADA_rank_AB = {
'BEZLOTOXUMAB':1,
'VISILIZUMAB':2,
'OMALIZUMAB':3,
'EVOLOCUMAB':4,
'SECUKINUMAB':5,
'DENOSUMAB':6,
'IBALIZUMAB':7,
'OCRELIZUMAB':8,
'FREMANEZUMAB':9,
'BASILIXIMAB':10,
'PALIVIZUMAB':11,
'CANAKINUMAB':12,
'ECULIZUMAB':13,
'BRODALUMAB':14,
'MEPOLIZUMAB':15,
'GUSELKUMAB':16,
'RESLIZUMAB':17,
'ALIROCUMAB':18,
'GALCANEZUMAB':19,
'VEDOLIZUMAB':20,
'EFALIZUMAB':21,
'TILDRAKIZUMAB':22,
'USTEKINUMAB':23,
'DUPILUMAB':24,
'ERENUMAB':25,
'SARILUMAB':26,
'NATALIZUMAB':27,
'LANADELUMAB':28,
'BUROSUMAB':29,
'BENRALIZUMAB':30,
'ADALIMUMAB':31,
'IXEKIZUMAB':32,
'RITUXIMAB':33,
'INFLIXIMAB':34,
'GOLIMUMAB':35,
'BOCOCIZUMAB':36,
'ALEMTUZUMAB':37
}

In [15]:
# Add the ADA rank to the dfs
# AntiBody df
for df in all_dfs_AB.values():
    df['ADA_rank'] = df['Antibody'].map(ADA_rank_AB)

# Nanobody df
for df in all_dfs.values():
    df['ADA_rank'] = df['Antibody'].map(ADA_rank)

In [16]:
# Compute MARE for the AB dataframes

# Add a column called MARE, for each row add the value for the absolute difference between the ranked_by_tool and ADA_rank
for df in all_dfs_AB.values():
    df['MARE'] = (df['ranked_by_tool'] - df['ADA_rank']).abs()

# Do the same thing for the dataframes with the nanobodies
for df in all_dfs.values():
    df['MARE'] = df['ranked_by_tool'] - df['ADA_rank'] 


In [17]:
# Create a ranked_score_table df with coulmns, datadframe (with all df names from all_dfs_AB),
# The sum of MARE for each df, 
# The spearman rank correlation between ranked by tool and ADA rank,
# The MARE for Caplacizumab and Vobarilizumab
ranked_score_table = pd.DataFrame({
    'dataframe': list(all_dfs_AB.keys()),
    'sum_MARE': [df['MARE'].sum() for df in all_dfs_AB.values()],
    'spearmanr': [spearmanr(df['ranked_by_tool'], df['ADA_rank']).correlation for df in all_dfs_AB.values()],
    'spearmanr_pval': [spearmanr(df['ranked_by_tool'], df['ADA_rank']).pvalue for df in all_dfs_AB.values()],
    'Caplacizumab_MARE': [df.loc[df['Antibody'] == 'Caplacizumab', 'MARE'].values[0] for df in all_dfs.values()],
    'Vobarilizumab_MARE': [df.loc[df['Antibody'] == 'Vobarilizumab', 'MARE'].values[0] for df in all_dfs.values()]
    
})

# save the ranked_score_table as a csv file
ranked_score_table.to_csv('ranked_score_table.csv', index=False)
ranked_score_table

,dataframe,sum_MARE,spearmanr,spearmanr_pval,Caplacizumab_MARE,Vobarilizumab_MARE
0,netMHC_II_immunogen_pepl11_allel27_wind2_AB,470.0,-0.124467,0.462963,-27.0,-28.0
1,netMHC_II_immunogen_pepl12_allel27_wind2_AB,458.0,-0.117591,0.488222,-25.0,-31.0
2,netMHC_II_immunogen_pepl12_allel27_wind5_AB,478.0,-0.100759,0.552935,-27.0,-31.0
3,netMHC_II_immunogen_pepl12_allel27_wind10_AB,392.0,0.124941,0.461247,4.0,-25.0
4,netMHC_II_immunogen_pepl15_allel27_wind2_AB,508.0,-0.250356,0.135044,-25.0,-29.0
5,netMHC_II_immunogen_pepl15_allel27_wind10_AB,498.0,-0.231152,0.168664,-15.0,-13.0
6,netMHC_II_immunogen_pepl15_allel27_wind30_AB,458.0,0.063300,0.709746,-6.0,-30.0
7,netMHC_II_immunogen_pepl15_allel7_wind5_AB,522.0,-0.281413,0.091544,-24.0,-27.0
8,netMHC_II_immunogen_pepl11_allel7_wind2_AB,470.0,-0.124467,0.462963,-27.0,-28.0
